# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/estuchis21/ml-internship-week1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

## My rule and its reason codes

### Rule

The baseline prioritizes content refresh opportunities using two observable signals:

1. Content freshness:
- Pages that have not been updated for more than 90 days receive a higher priority score.

2. Search visibility:
- Pages with impressions_last_30d above the 65th percentile are considered higher-value because they still receive search visibility.

The idea behind the rule is:

Old content + existing search visibility = potential refresh opportunity.

This is a prioritization rule, not a prediction. It identifies pages that deserve human review.

### Reason codes

STALE_HIGH_VALUE:
The content is old and still receives a significant number of impressions. It may represent an opportunity to improve or refresh the content.

LOW_PRIORITY:
The content does not satisfy the conditions for immediate refresh review.

### Actions

REFRESH_CONTENT:
Assigned when the content is old and has high visibility.

REVIEW_CONTENT:
Assigned when the content is old but does not have enough evidence of high value.

NO_ACTION:
Assigned when the content does not show strong refresh signals.

In [60]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

url = "https://raw.githubusercontent.com/estuchis21/ml-internship-week1/refs/heads/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)
df.columns

signals_df = df[
    [
        "content_id",
        "trend_pct",
        "days_since_last_update",
        "impressions_last_30d"
    ]
].copy()


signals_df.head()


,content_id,trend_pct,days_since_last_update,impressions_last_30d
0,content_304f48230142,-41.4,20,578
1,content_a1fb4e703a9e,-57.7,25,2501
2,content_9aa793d4d895,-60.9,20,2382
3,content_331d6c4de07b,-13.8,22,3626
4,content_d99b7a2d90ca,-34.7,14,4211


## 2. Build the ranked queue (writes the CSV)
## Signal validation

The two signals selected for validation are:

### Signal 1: days_since_last_update

Reason:
Content freshness is directly related to refresh opportunities. Older pages may become outdated and could benefit from updates.

Validation approach:
Pages are grouped into freshness buckets and compared using average impressions_last_30d.

Verdict:
CONFIRMED

The results show that older content groups can still contain pages with meaningful impressions, supporting the idea that stale pages may represent valuable refresh candidates.

---

### Signal 2: trend_pct

Reason:
A declining trend may indicate that content is losing visibility and could benefit from optimization.

Validation approach:
Pages are grouped according to trend percentage ranges and compared using average impressions_last_30d.

Verdict:
MIXED

Some declining pages still receive impressions, meaning a negative trend can indicate an opportunity, but it does not guarantee that refreshing content will improve performance.

In [73]:
# Crear buckets de days_since_last_update

# Crear buckets de trend_pct

signals_df["trend_bucket"] = np.select(
    [
        signals_df["trend_pct"] <= -40,
        (signals_df["trend_pct"] > -40) &
        (signals_df["trend_pct"] <= -20),
        (signals_df["trend_pct"] > -20) &
        (signals_df["trend_pct"] <= 0),
        (signals_df["trend_pct"] > 0) &
        (signals_df["trend_pct"] <= 20),
        signals_df["trend_pct"] > 20
    ],
    [
        "very_negative",
        "negative",
        "neutral",
        "positive",
        "very_positive"
    ],
    default="unknown"
)

# Tabla con cantidad de páginas por bucket
trend_table = signals_df.groupby("trend_bucket").agg(
    n=("trend_pct","count"),
    avg_impressions=("impressions_last_30d","mean")
)

signals_df["update_bucket"] = np.select(
    [
        signals_df["days_since_last_update"] <= 30,

        (signals_df["days_since_last_update"] > 30) &
        (signals_df["days_since_last_update"] <= 90),

        (signals_df["days_since_last_update"] > 90) &
        (signals_df["days_since_last_update"] <= 180),

        signals_df["days_since_last_update"] > 180
    ],
    [
        "recent_update",
        "medium_update",
        "old_update",
        "very_old_update"
    ],
    default="unknown"
)

update_table = signals_df.groupby("update_bucket").agg(
    n=("days_since_last_update","count"),
    avg_impressions=("impressions_last_30d","mean")
)

print(update_table)
print(trend_table)


                   n  avg_impressions
trend_bucket                         
negative        4446      1897.667341
neutral         3845      2914.676723
positive        2068      3114.572534
unknown            0       105.896104
very_negative  11867       581.398247
very_positive   4386      2172.511628
                     n  avg_impressions
update_bucket                          
medium_update      175      2120.085714
old_update        9171      2001.390906
recent_update    20480      1178.084326
very_old_update    174       108.183908
                   n  avg_impressions
trend_bucket                         
negative        4446      1897.667341
neutral         3845      2914.676723
positive        2068      3114.572534
unknown            0       105.896104
very_negative  11867       581.398247
very_positive   4386      2172.511628


## 3. Top-20 review

## Top-20 review

The ranked queue prioritizes pages with the STALE_HIGH_VALUE reason code.

Each selected page has:
- old content age
- meaningful search impressions

These pages are considered refresh candidates.

The confidence is medium because the rule uses only two signals and does not consider all possible causes of traffic changes.

Possible reasons the recommendation could be wrong:
- Traffic may be seasonal.
- The page may already satisfy user intent.
- Updating the content may not improve rankings.
- The page may be receiving impressions without meaningful opportunity.

Example review format:

Content ID: content_887020f20b5e

Action:
REFRESH_CONTENT

Reason code:
STALE_HIGH_VALUE

Confidence:
Medium confidence because the page is old and still receives search impressions.

What would make it wrong:
The traffic could be stable or seasonal, meaning a refresh might not create additional value.

---

Content ID: content_b5db993ce36a

Action:
REFRESH_CONTENT

Reason code:
STALE_HIGH_VALUE

Confidence:
Medium confidence because the page has historical visibility and has not been updated recently.

What would make it wrong:
The content could already be performing well without changes.

---

Content ID: content_1db0d204d42f

Action:
REFRESH_CONTENT

Reason code:
STALE_HIGH_VALUE

Confidence:
Medium confidence because the page combines freshness risk with search visibility.

What would make it wrong:
The decline may not be related to content freshness.

In [72]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
baseline = df.copy()


baseline["score"] = 0


# contenido viejo
baseline.loc[
    baseline["days_since_last_update"] > 90,
    "score"
] += 50


# mucho tráfico
baseline.loc[
    baseline["impressions_last_30d"] >= baseline["impressions_last_30d"].quantile(0.65),
    "score"
] += 30


baseline["reason_code"] = "LOW_PRIORITY"


baseline.loc[
    (baseline["days_since_last_update"] > 90) &
    (baseline["impressions_last_30d"] >= baseline["impressions_last_30d"].quantile(0.65)),
    "reason_code"
] = "STALE_HIGH_VALUE"


baseline["action_label"] = "NO_ACTION"


baseline.loc[
    baseline["score"] >= 80,
    "action_label"
] = "REFRESH_CONTENT"


baseline.loc[
    (baseline["score"] >= 50) &
    (baseline["score"] < 80),
    "action_label"
] = "REVIEW_CONTENT"


baseline_queue = baseline.sort_values(
    "score",
    ascending=False
)
from pathlib import Path

Path("work/outputs").mkdir(parents=True, exist_ok=True)

baseline_queue[
    [
        "content_id",
        "score",
        "reason_code",
        "action_label",
        "days_since_last_update",
        "impressions_last_30d"
    ]
].to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

baseline_queue['action_label'].value_counts()

print(baseline_queue.head(20))

                 content_id          client_id  search_volume  competition  \
29999  content_887020f20b5e  client_6208ef0f77            0.0         0.00   
29969  content_b5db993ce36a  client_6208ef0f77            0.0         0.00   
29972  content_1db0d204d42f  client_6208ef0f77            0.0         0.00   
29982  content_fe5d259e6bc5  client_19581e27de           20.0         0.10   
21279  content_05f46b36f1b2  client_19581e27de           10.0         0.00   
9345   content_c22c1dd7122d  client_6208ef0f77            0.0         0.00   
9348   content_3430a8b94511  client_19581e27de           10.0         0.68   
9355   content_7d428d4b9435  client_6208ef0f77            0.0         0.00   
9365   content_37d52a7d751c  client_6208ef0f77            0.0         0.00   
29931  content_31b21583c35a  client_19581e27de           10.0         0.00   
29933  content_a219938761d0  client_19581e27de           10.0         0.10   
55     content_a519180b7a3f  client_6208ef0f77            0.0   

## Weak picks + leakage check

### Weak picks

Some high-scoring pages may be false positives.

Possible reasons:

- A page may have high impressions because of seasonality or temporary demand.
- A page may already satisfy user needs even if it is old.
- Updating content does not guarantee better rankings or traffic.

For example, pages with increasing trends may be weaker refresh candidates because their performance is already improving.

### Leakage check

The baseline uses only historical observable signals:

- days_since_last_update
- impressions_last_30d

These variables are available before making the refresh decision.

The baseline does not use:

- FlyRank product flags.
- Labels.
- Future windows.
- Any target-derived information.

Therefore, no obvious data leakage was introduced.

In [70]:
# Tomar contenidos con mayor prioridad
weak_candidates = baseline_queue[
    baseline_queue["action_label"] == "REFRESH_CONTENT"
]


# Mostrar algunos ejemplos para revisar manualmente
weak_picks = weak_candidates.head(10)


weak_picks[
    [
        "content_id",
        "score",
        "reason_code",
        "action_label",
        "days_since_last_update",
        "impressions_last_30d"
    ]
]

# Ver columnas disponibles

df.columns.tolist()

# Buscar posibles columnas con leakage

possible_leakage = [
    col for col in df.columns
    if (
        "flag" in col.lower()
        or "label" in col.lower()
        or "target" in col.lower()
        or "future" in col.lower()
        or "next" in col.lower()
        or "outcome" in col.lower()
    )
]


# Features utilizadas por la regla

baseline_features = [
    "days_since_last_update",
    "impressions_last_30d"
]


baseline_features
# Confirmación simple

used_columns = baseline_features


for col in used_columns:
    print(col, "OK")


days_since_last_update OK
impressions_last_30d OK


In [74]:
X = df[
    [
        "days_since_last_update",
        "impressions_last_30d"
    ]
]

df["refresh_flag"] = (
    (df["days_since_last_update"] > 90) &
    (df["impressions_last_30d"] >= df["impressions_last_30d"].quantile(0.65))
).astype(int)

y = df["refresh_flag"]

In [78]:
from sklearn.dummy import DummyClassifier


dummy = DummyClassifier(
    strategy="prior"
)

dummy.fit(X,y)

pred = dummy.predict(X)

pred[:10]

from sklearn.metrics import classification_report


print(
    classification_report(
        y,
        pred
    )
)

              precision    recall  f1-score   support

           0       0.86      1.00      0.92     25763
           1       0.00      0.00      0.00      4237

    accuracy                           0.86     30000
   macro avg       0.43      0.50      0.46     30000
weighted avg       0.74      0.86      0.79     30000



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.